In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# CREDRESOLVE — STAGE 8: INVESTMENT CASE
# ============================================================
# Uses corrected Stage 7 outputs.
#
# Recovery amount:
#   SUCCESS payments only
#
# Investment scenario:
#   ₹10 Cr
#
# IMPORTANT:
#   Modeled uplift is observational/model-based.
#   It is NOT treated as causal proof.
# ============================================================

PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("CREDRESOLVE — STAGE 8: INVESTMENT CASE")
print("=" * 80)


# ============================================================
# 1. LOAD STAGE 7 OUTPUTS
# ============================================================

baseline = pd.read_csv(
    OUTPUT_DIR / "counterfactual_baseline.csv"
)

scenarios = pd.read_csv(
    OUTPUT_DIR / "counterfactual_scenarios.csv"
)

sensitivity_stage7 = pd.read_csv(
    OUTPUT_DIR / "counterfactual_sensitivity.csv"
)

print("Stage 7 outputs loaded successfully.")
print()


# ============================================================
# 2. VALIDATE REQUIRED COLUMNS
# ============================================================

required_baseline_columns = [
    "accounts",
    "recovered_accounts",
    "unrecovered_accounts",
    "baseline_recovery_rate",
    "baseline_recovery_amount",
    "baseline_outstanding_amount",
    "average_recovery_per_recovered_account"
]

missing = [
    c for c in required_baseline_columns
    if c not in baseline.columns
]

if missing:
    raise ValueError(
        f"Missing baseline columns: {missing}"
    )


required_scenario_columns = [
    "scenario",
    "description",
    "driver",
    "model_effect_pp",
    "eligible_accounts",
    "estimated_incremental_recovery_amount",
    "estimated_incremental_recovered_accounts"
]

missing = [
    c for c in required_scenario_columns
    if c not in scenarios.columns
]

if missing:
    raise ValueError(
        f"Missing scenario columns: {missing}"
    )


# ============================================================
# 3. BASELINE
# ============================================================

accounts = int(
    baseline.loc[0, "accounts"]
)

recovered_accounts = int(
    baseline.loc[0, "recovered_accounts"]
)

unrecovered_accounts = int(
    baseline.loc[0, "unrecovered_accounts"]
)

baseline_rate = float(
    baseline.loc[0, "baseline_recovery_rate"]
)

baseline_amount = float(
    baseline.loc[0, "baseline_recovery_amount"]
)

outstanding_amount = float(
    baseline.loc[0, "baseline_outstanding_amount"]
)

average_recovery = float(
    baseline.loc[
        0,
        "average_recovery_per_recovered_account"
    ]
)


# ============================================================
# 4. VALIDATE CORRECTED STAGE 7 BASELINE
# ============================================================

if accounts != 30000:
    raise ValueError(
        f"Expected 30,000 accounts, found {accounts}"
    )

if recovered_accounts != 13284:
    raise ValueError(
        f"Expected 13,284 recovered accounts, "
        f"found {recovered_accounts}"
    )

if abs(baseline_rate - 0.4428) > 0.0001:
    raise ValueError(
        f"Expected recovery rate near 0.4428, "
        f"found {baseline_rate}"
    )

EXPECTED_RECOVERY_AMOUNT = 1_315_584_000

if abs(
    baseline_amount - EXPECTED_RECOVERY_AMOUNT
) > 5000:

    raise ValueError(
        "\nINCORRECT RECOVERY AMOUNT.\n"
        f"Found: ₹{baseline_amount:,.2f}\n"
        f"Expected: approximately "
        f"₹{EXPECTED_RECOVERY_AMOUNT:,.2f}\n"
        "\nFix Stage 7 before continuing."
    )


baseline_summary = pd.DataFrame([{

    "accounts_analyzed":
        accounts,

    "recovered_accounts":
        recovered_accounts,

    "unrecovered_accounts":
        unrecovered_accounts,

    "baseline_recovery_rate":
        baseline_rate,

    "baseline_recovery_amount":
        baseline_amount,

    "baseline_outstanding_amount":
        outstanding_amount,

    "average_recovery_per_recovered_account":
        average_recovery,

    "recovery_definition":
        "SUCCESS payments only after payment_id deduplication"

}])


print("=" * 80)
print("CORRECTED BASELINE")
print("=" * 80)

display(baseline_summary)


# ============================================================
# 5. STAGE 7 SCENARIOS
# ============================================================

scenario_output = scenarios.copy()

scenario_output[
    "incremental_recovery_value"
] = pd.to_numeric(
    scenario_output[
        "estimated_incremental_recovery_amount"
    ],
    errors="coerce"
).fillna(0)

scenario_output[
    "incremental_recovered_accounts"
] = pd.to_numeric(
    scenario_output[
        "estimated_incremental_recovered_accounts"
    ],
    errors="coerce"
).fillna(0)

scenario_output[
    "model_effect_pp"
] = pd.to_numeric(
    scenario_output[
        "model_effect_pp"
    ],
    errors="coerce"
).fillna(0)

scenario_output[
    "incremental_recovery_vs_baseline_pct"
] = (

    scenario_output[
        "incremental_recovery_value"
    ]
    /
    baseline_amount
    *
    100

)

if "causal_claim" not in scenario_output.columns:

    scenario_output[
        "causal_claim"
    ] = False


print("=" * 80)
print("STAGE 7 SCENARIOS")
print("=" * 80)

display(
    scenario_output[
        [
            "scenario",
            "description",
            "driver",
            "model_effect_pp",
            "eligible_accounts",
            "incremental_recovered_accounts",
            "incremental_recovery_value",
            "incremental_recovery_vs_baseline_pct",
            "causal_claim"
        ]
    ]
)


# ============================================================
# 6. FIND TELEPHONY SCENARIO
# ============================================================
#
# IMPORTANT:
#
# Scenario A = Conservative
# Driver     = Telephony
#
# Therefore search DRIVER, not scenario name.
# ============================================================

telephony_mask = (

    scenario_output[
        "driver"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("telephony")

)


if not telephony_mask.any():

    # fallback in case the value contains extra text

    telephony_mask = (

        scenario_output[
            "driver"
        ]
        .astype(str)
        .str.contains(
            "telephony",
            case=False,
            na=False
        )

    )


if not telephony_mask.any():

    raise ValueError(
        "\nTELEPHONY SCENARIO NOT FOUND.\n"
        "Available drivers:\n"
        +
        "\n".join(
            scenario_output[
                "driver"
            ].astype(str).unique()
        )
    )


telephony_row = (

    scenario_output[
        telephony_mask
    ]
    .iloc[0]

)


# ============================================================
# 7. TELEPHONY MODEL VALUES
# ============================================================

telephony_scenario = (
    telephony_row["scenario"]
)

telephony_driver = (
    telephony_row["driver"]
)

telephony_model_effect_pp = float(
    telephony_row["model_effect_pp"]
)

telephony_incremental_recovery = float(
    telephony_row[
        "incremental_recovery_value"
    ]
)

telephony_incremental_accounts = float(
    telephony_row[
        "incremental_recovered_accounts"
    ]
)


print("=" * 80)
print("TELEPHONY SCENARIO FOUND")
print("=" * 80)

print(
    f"Scenario: {telephony_scenario}"
)

print(
    f"Driver: {telephony_driver}"
)

print(
    f"Model effect: "
    f"{telephony_model_effect_pp:.6f} pp"
)

print(
    f"Incremental recovery: "
    f"₹{telephony_incremental_recovery:,.2f}"
)

print(
    f"Incremental recovered accounts: "
    f"{telephony_incremental_accounts:,.0f}"
)


# ============================================================
# 8. ₹10 CR INVESTMENT
# ============================================================

INVESTMENT_AMOUNT = 100_000_000


# ============================================================
# 9. BREAK-EVEN
# ============================================================

if outstanding_amount <= 0:
    raise ValueError(
        "Outstanding amount must be greater than zero."
    )


break_even_uplift_pp = (

    INVESTMENT_AMOUNT
    /
    outstanding_amount
    *
    100

)


# ============================================================
# 10. TELEPHONY EFFECT
# ============================================================
#
# Use Stage 7 model_effect_pp directly.
#
# This avoids accidentally losing the modeled effect.
# ============================================================

telephony_effect_pp = (
    telephony_model_effect_pp
)

telephony_margin_vs_break_even_pp = (

    telephony_effect_pp
    -
    break_even_uplift_pp

)


# ============================================================
# 11. ECONOMIC STATUS
# ============================================================

if (
    telephony_effect_pp
    >=
    break_even_uplift_pp
):

    telephony_economic_status = (
        "MODELED EFFECT CLEARS BREAK-EVEN"
    )

else:

    telephony_economic_status = (
        "MODELED EFFECT DOES NOT CLEAR BREAK-EVEN"
    )


print("=" * 80)
print("₹10 CR BREAK-EVEN ANALYSIS")
print("=" * 80)

print(
    f"Investment: ₹{INVESTMENT_AMOUNT:,.0f}"
)

print(
    f"Outstanding portfolio: "
    f"₹{outstanding_amount:,.2f}"
)

print(
    f"Break-even uplift: "
    f"{break_even_uplift_pp:.6f} pp"
)

print(
    f"Telephony modeled effect: "
    f"{telephony_effect_pp:.6f} pp"
)

print(
    f"Margin vs break-even: "
    f"{telephony_margin_vs_break_even_pp:.6f} pp"
)

print(
    f"Telephony incremental recovery: "
    f"₹{telephony_incremental_recovery:,.2f}"
)

print(
    f"Status: {telephony_economic_status}"
)


# ============================================================
# 12. BREAK-EVEN OUTPUT
# ============================================================

break_even_summary = pd.DataFrame([{

    "investment_amount":
        INVESTMENT_AMOUNT,

    "portfolio_outstanding":
        outstanding_amount,

    "break_even_uplift_pp":
        break_even_uplift_pp,

    "telephony_modeled_effect_pp":
        telephony_effect_pp,

    "telephony_margin_vs_break_even_pp":
        telephony_margin_vs_break_even_pp,

    "telephony_incremental_recovery_value":
        telephony_incremental_recovery,

    "telephony_incremental_recovered_accounts":
        telephony_incremental_accounts,

    "investment_cost_source":
        "₹10 Cr assignment scenario — not observed source data",

    "incremental_effect_source":
        "Stage 7 counterfactual/model estimate",

    "causal_effect_established":
        False

}])


display(
    break_even_summary
)


# ============================================================
# 13. DOWNSIDE / BASE / UPSIDE
# ============================================================

sensitivity_rows = []


sensitivity_assumptions = {

    "Downside": 0.50,

    "Base": 1.00,

    "Upside": 1.25

}


for scenario_name, multiplier in (
    sensitivity_assumptions.items()
):

    assumed_uplift_pp = (

        telephony_effect_pp
        *
        multiplier

    )

    incremental_recovery = (

        outstanding_amount
        *
        assumed_uplift_pp
        /
        100

    )

    net_value = (

        incremental_recovery
        -
        INVESTMENT_AMOUNT

    )

    roi_pct = (

        net_value
        /
        INVESTMENT_AMOUNT
        *
        100

    )

    sensitivity_rows.append({

        "scenario":
            scenario_name,

        "investment_amount":
            INVESTMENT_AMOUNT,

        "assumed_uplift_pp":
            assumed_uplift_pp,

        "incremental_recovery":
            incremental_recovery,

        "net_value_after_investment":
            net_value,

        "roi_pct":
            roi_pct,

        "meets_break_even":
            incremental_recovery >= INVESTMENT_AMOUNT,

        "scenario_type":
            "Sensitivity assumption — not observed fact",

        "causal_claim":
            False

    })


investment_sensitivity = pd.DataFrame(
    sensitivity_rows
)


print("=" * 80)
print("₹10 CR DOWNSIDE / BASE / UPSIDE")
print("=" * 80)

display(
    investment_sensitivity
)


# ============================================================
# 14. HYPOTHETICAL ROI SENSITIVITY
# ============================================================

cost_levels = [

    5_000_000,
    10_000_000,
    20_000_000,
    30_000_000,
    50_000_000

]


roi_rows = []


for _, row in scenario_output.iterrows():

    scenario_name = row[
        "scenario"
    ]

    value = float(
        row[
            "incremental_recovery_value"
        ]
    )

    for cost in cost_levels:

        net_value = (
            value - cost
        )

        roi = (

            net_value
            /
            cost

            if cost > 0
            else np.nan

        )

        roi_rows.append({

            "scenario":
                scenario_name,

            "hypothetical_cost":
                cost,

            "incremental_recovery_value":
                value,

            "net_value":
                net_value,

            "roi_pct":
                roi * 100,

            "cost_type":
                "Hypothetical sensitivity — not observed cost"

        })


roi_df = pd.DataFrame(
    roi_rows
)


# ============================================================
# 15. BUSINESS CASE
# ============================================================

case_map = {

    "A": "Conservative",

    "B": "Targeted Opportunity",

    "C": "Upside"

}


business_case = scenario_output[

    [
        "scenario",
        "description",
        "driver",
        "eligible_accounts",
        "incremental_recovered_accounts",
        "incremental_recovery_value",
        "causal_claim"
    ]

].copy()


business_case[
    "business_case"
] = (

    business_case[
        "scenario"
    ].map(case_map)

)


business_case = business_case[

    [
        "business_case",
        "scenario",
        "description",
        "driver",
        "eligible_accounts",
        "incremental_recovered_accounts",
        "incremental_recovery_value",
        "causal_claim"
    ]

]


# ============================================================
# 16. ASSUMPTIONS
# ============================================================

assumptions = pd.DataFrame([

    {
        "assumption":
            "Recovery definition",

        "value":
            "SUCCESS payments only",

        "interpretation":
            "Payment IDs deduplicated before aggregation"
    },

    {
        "assumption":
            "Investment",

        "value":
            "₹10 Cr",

        "interpretation":
            "Assignment scenario, not observed cost"
    },

    {
        "assumption":
            "Telephony effect",

        "value":
            f"{telephony_effect_pp:.6f} pp",

        "interpretation":
            "Stage 7 modeled estimate, not causal proof"
    },

    {
        "assumption":
            "Break-even",

        "value":
            f"{break_even_uplift_pp:.6f} pp",

        "interpretation":
            "₹10 Cr divided by portfolio outstanding"
    },

    {
        "assumption":
            "Implementation cost",

        "value":
            "Not available from supplied data",

        "interpretation":
            "No actual cost invented"
    },

    {
        "assumption":
            "ROI",

        "value":
            "Sensitivity analysis only",

        "interpretation":
            "Not realized ROI"
    },

    {
        "assumption":
            "Causal effect",

        "value":
            "Not established",

        "interpretation":
            "Controlled pilot required"
    }

])


# ============================================================
# 17. FINAL RECOMMENDATION
# ============================================================

recommendation = (

    "Pilot first — do not commit the full ₹10 Cr "
    "until incremental recovery is validated."

)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

best_index = (

    scenario_output[
        "incremental_recovery_value"
    ].idxmax()

)

best = scenario_output.loc[
    best_index
]


final_summary = pd.DataFrame([{

    "accounts_analyzed":
        accounts,

    "baseline_recovery_rate":
        baseline_rate,

    "baseline_recovery_amount":
        baseline_amount,

    "baseline_outstanding_amount":
        outstanding_amount,

    "investment_amount":
        INVESTMENT_AMOUNT,

    "break_even_uplift_pp":
        break_even_uplift_pp,

    "telephony_modeled_effect_pp":
        telephony_effect_pp,

    "telephony_margin_vs_break_even_pp":
        telephony_margin_vs_break_even_pp,

    "telephony_incremental_recovery_value":
        telephony_incremental_recovery,

    "telephony_economic_status":
        telephony_economic_status,

    "largest_scenario":
        best["scenario"],

    "largest_scenario_incremental_recovery_value":
        best["incremental_recovery_value"],

    "actual_implementation_cost_available":
        False,

    "roi_claim_made":
        False,

    "causal_claim_made":
        False,

    "confidence":
        "Low–Moderate",

    "evidence":
        "Observational / model-based",

    "recommended_action":
        recommendation

}])


print("=" * 80)
print("FINAL INVESTMENT CASE")
print("=" * 80)

display(final_summary)


# ============================================================
# 19. SAVE ALL OUTPUTS
# ============================================================

baseline_summary.to_csv(
    OUTPUT_DIR / "investment_baseline_metrics.csv",
    index=False
)

scenario_output.to_csv(
    OUTPUT_DIR / "investment_scenario_economics.csv",
    index=False
)

break_even_summary.to_csv(
    OUTPUT_DIR / "investment_10cr_break_even.csv",
    index=False
)

investment_sensitivity.to_csv(
    OUTPUT_DIR / "investment_10cr_sensitivity.csv",
    index=False
)

roi_df.to_csv(
    OUTPUT_DIR / "investment_roi_sensitivity.csv",
    index=False
)

business_case.to_csv(
    OUTPUT_DIR / "investment_business_case.csv",
    index=False
)

assumptions.to_csv(
    OUTPUT_DIR / "investment_assumptions.csv",
    index=False
)

final_summary.to_csv(
    OUTPUT_DIR / "investment_case_summary.csv",
    index=False
)


# ============================================================
# 20. FINAL STATUS
# ============================================================

print("=" * 80)
print("INVESTMENT CASE COMPLETE")
print("=" * 80)

print(
    f"Corrected recovery amount: "
    f"₹{baseline_amount:,.2f}"
)

print(
    f"Telephony modeled effect: "
    f"{telephony_effect_pp:.6f} pp"
)

print(
    f"₹10 Cr break-even: "
    f"{break_even_uplift_pp:.6f} pp"
)

print(
    f"Margin vs break-even: "
    f"{telephony_margin_vs_break_even_pp:.6f} pp"
)

print(
    f"Telephony incremental recovery: "
    f"₹{telephony_incremental_recovery:,.2f}"
)

print(
    "Causal claims made: FALSE"
)

print(
    "Implementation cost invented: FALSE"
)

print(
    "Recommended action:"
)

print(
    recommendation
)

CREDRESOLVE — STAGE 8: INVESTMENT CASE
Stage 7 outputs loaded successfully.

CORRECTED BASELINE


,accounts_analyzed,recovered_accounts,unrecovered_accounts,baseline_recovery_rate,baseline_recovery_amount,baseline_outstanding_amount,average_recovery_per_recovered_account,recovery_definition
0,30000,13284,16716,0.4428,1.315584e+09,1.048904e+10,99035.22769,SUCCESS payments only after payment_id dedupli...


STAGE 7 SCENARIOS


,scenario,description,driver,model_effect_pp,eligible_accounts,incremental_recovered_accounts,incremental_recovery_value,incremental_recovery_vs_baseline_pct,causal_claim
0,A,Conservative,Telephony,1.076249,30000,322.874703,1.128881e+08,8.580839,False
1,B,Targeted Opportunity,Collection Agents,0.644541,30000,193.362358,6.760615e+07,5.138870,False
2,C,Upside,Borrower Targeting,0.000000,30000,0.000000,0.000000e+00,0.000000,False


TELEPHONY SCENARIO FOUND
Scenario: A
Driver: Telephony
Model effect: 1.076249 pp
Incremental recovery: ₹112,888,139.21
Incremental recovered accounts: 323
₹10 CR BREAK-EVEN ANALYSIS
Investment: ₹100,000,000
Outstanding portfolio: ₹10,489,035,343.00
Break-even uplift: 0.953377 pp
Telephony modeled effect: 1.076249 pp
Margin vs break-even: 0.122872 pp
Telephony incremental recovery: ₹112,888,139.21
Status: MODELED EFFECT CLEARS BREAK-EVEN


...

₹10 CR DOWNSIDE / BASE / UPSIDE


,scenario,investment_amount,assumed_uplift_pp,incremental_recovery,net_value_after_investment,roi_pct,meets_break_even,scenario_type,causal_claim
0,Downside,100000000,0.538125,5.644407e+07,-4.355593e+07,-43.555930,False,Sensitivity assumption — not observed fact,False
1,Base,100000000,1.076249,1.128881e+08,1.288814e+07,12.888139,True,Sensitivity assumption — not observed fact,False
2,Upside,100000000,1.345311,1.411102e+08,4.111017e+07,41.110174,True,Sensitivity assumption — not observed fact,False


FINAL INVESTMENT CASE


,accounts_analyzed,baseline_recovery_rate,baseline_recovery_amount,baseline_outstanding_amount,investment_amount,break_even_uplift_pp,telephony_modeled_effect_pp,telephony_margin_vs_break_even_pp,telephony_incremental_recovery_value,telephony_economic_status,largest_scenario,largest_scenario_incremental_recovery_value,actual_implementation_cost_available,roi_claim_made,causal_claim_made,confidence,evidence,recommended_action
0,30000,0.4428,1.315584e+09,1.048904e+10,100000000,0.953377,1.076249,0.122872,1.128881e+08,MODELED EFFECT CLEARS BREAK-EVEN,A,1.128881e+08,False,False,False,Low–Moderate,Observational / model-based,Pilot first — do not commit the full ₹10 Cr un...


INVESTMENT CASE COMPLETE
Corrected recovery amount: ₹1,315,583,964.64
Telephony modeled effect: 1.076249 pp
₹10 Cr break-even: 0.953377 pp
Margin vs break-even: 0.122872 pp
Telephony incremental recovery: ₹112,888,139.21
Causal claims made: FALSE
Implementation cost invented: FALSE
Recommended action:
Pilot first — do not commit the full ₹10 Cr until incremental recovery is validated.
